# 05 - Grad-CAM temporal explainability evaluation

This notebook loads trained 2D U-Net and ConvLSTM U-Net checkpoints and computes Grad-CAM-style temporal explanations on the official EchoNet-Dynamic test split only. It does not train models.

In [ ]:
# Kaggle setup. Skip this cell when the environment already satisfies requirements.txt.
%pip install -q monai opencv-python-headless pandas matplotlib tqdm

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import split_by_echonet_filelist
from src.gradcam import convlstm_gradcam, find_last_conv2d_name, unet_framewise_gradcam
from src.model import build_unet
from src.temporal_dataset_variable_stride import (
    EchoNetTemporalVariableStrideDataset,
    build_fps_lookup,
    load_temporal_metadata,
)
from src.temporal_evaluation import (
    aggregate_metrics,
    compute_temporal_saliency_metrics,
)
from src.temporal_model import build_convlstm_unet
from src.utils import load_echonet_tables, set_seed
from src.visualization import sanitize_name, save_heatmaps, save_metric_plots, save_overlay_grid

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

In [ ]:
RUN_MODE = 'smoke'  # use 'full' for the entire official test set
SMOKE_RANDOM_SEED = 42
SALIENCY_PERCENTILE = 80.0
MAX_OVERLAY_SAMPLES = 1 if RUN_MODE == 'smoke' else 20

RAW_DIR = Path(os.environ.get('ECHONET_RAW_DIR', PROJECT_ROOT / 'data' / 'raw' / 'EchoNet-Dynamic'))
PROCESSED_DIR = Path(os.environ.get('ECHONET_PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed'))
VIDEOS_DIR = RAW_DIR / 'Videos'
OUTPUT_DIR = Path('/kaggle/working/outputs/runs/gradcam_temporal_evaluation') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'gradcam_temporal_evaluation'

HEATMAP_DIR = OUTPUT_DIR / 'heatmaps'
OVERLAY_DIR = OUTPUT_DIR / 'overlays'
METRICS_DIR = OUTPUT_DIR / 'metrics'
FIGURES_DIR = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'tables'
for directory in [HEATMAP_DIR, OVERLAY_DIR, METRICS_DIR, FIGURES_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONVLSTM_TARGET_LAYERS = ['bottleneck_encoder', 'temporal_bottleneck', 'decoder3']
CONVLSTM_STRIDES = [1, 4, 6, 8, 10]
SEQUENCE_LENGTH = 5
IMAGE_SIZE = (112, 112)
BATCH_SIZE = 1
NUM_WORKERS = 0

print(f'Raw data: {RAW_DIR}')
print(f'Processed data: {PROCESSED_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Max unique samples with saved overlays: {MAX_OVERLAY_SAMPLES}')

## Checkpoint discovery

In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


VARIABLE_STRIDE_ROOT = first_existing(
    [
        Path(os.environ.get('CONVLSTM_VARIABLE_STRIDE_RUN_DIR', '')),
        PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet_variable_strides_06_21',
        PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet_variable_strides',
    ]
)
STRIDE1_ROOT = first_existing(
    [
        Path(os.environ.get('CONVLSTM_STRIDE1_RUN_DIR', '')),
        PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet',
        PROJECT_ROOT / 'outputs' / 'runs' / 'ConvLSTM_Unet_06_11',
    ]
)
UNET_CHECKPOINT_PATH = first_existing(
    [
        Path(os.environ.get('UNET_CHECKPOINT_PATH', '')),
        PROJECT_ROOT / 'outputs' / 'checkpoints' / 'best_unet.pt',
        PROJECT_ROOT / 'outputs' / 'runs' / 'unet' / 'checkpoints' / 'best_unet.pt',
    ]
)

def convlstm_checkpoint_for_stride(stride: int):
    if stride == 1 and STRIDE1_ROOT is not None:
        return STRIDE1_ROOT / 'checkpoints' / 'best_model.pt'
    if stride != 1 and VARIABLE_STRIDE_ROOT is not None:
        return VARIABLE_STRIDE_ROOT / f'convlstm_unet_stride_{stride}' / 'checkpoints' / 'best_model.pt'
    return None


def config_for_stride(stride: int):
    if stride == 1 and STRIDE1_ROOT is not None:
        config_path = STRIDE1_ROOT / 'config.json'
    elif VARIABLE_STRIDE_ROOT is not None:
        config_path = VARIABLE_STRIDE_ROOT / f'convlstm_unet_stride_{stride}' / 'config.json'
    else:
        config_path = None
    if config_path and config_path.exists():
        with config_path.open('r', encoding='utf-8') as file:
            return json.load(file)
    return {'channels': [16, 32, 64, 128], 'sequence_length': SEQUENCE_LENGTH, 'image_size': list(IMAGE_SIZE)}


checkpoint_rows = []
for stride in CONVLSTM_STRIDES:
    path = convlstm_checkpoint_for_stride(stride)
    exists = bool(path and path.exists())
    checkpoint_rows.append({'model_family': 'convlstm_unet', 'stride': stride, 'checkpoint_path': str(path), 'exists': exists})
checkpoint_rows.append({'model_family': 'unet_2d', 'stride': None, 'checkpoint_path': str(UNET_CHECKPOINT_PATH), 'exists': bool(UNET_CHECKPOINT_PATH and UNET_CHECKPOINT_PATH.exists())})
checkpoint_df = pd.DataFrame(checkpoint_rows)
checkpoint_df.to_csv(TABLES_DIR / 'checkpoint_discovery.csv', index=False)
checkpoint_df

## Official test split only

In [ ]:
metadata_path = PROCESSED_DIR / 'metadata.csv'
assert metadata_path.exists(), 'metadata.csv is required from notebook 02 preprocessing.'
assert VIDEOS_DIR.exists(), f'Videos directory not found: {VIDEOS_DIR}'

samples = load_temporal_metadata(metadata_path)
file_list, _ = load_echonet_tables(RAW_DIR)
fps_by_video = build_fps_lookup(file_list)
_, _, test_samples = split_by_echonet_filelist(samples, file_list)
assert len(test_samples) > 0, 'Official EchoNet test split has no matched processed samples.'

if RUN_MODE == 'smoke':
    rng = random.Random(SMOKE_RANDOM_SEED)
    eval_samples = [rng.choice(test_samples)]
else:
    eval_samples = test_samples

print(f'Official test samples available: {len(test_samples):,}')
print(f'Evaluation samples selected for {RUN_MODE}: {len(eval_samples):,}')

## Model loading helpers

In [ ]:
def load_state_dict_from_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def build_convlstm_for_stride(stride: int):
    config = config_for_stride(stride)
    model = build_convlstm_unet(
        in_channels=1,
        out_channels=1,
        channels=tuple(config.get('channels', [16, 32, 64, 128])),
    ).to(device)
    return model, config


def build_unet_baseline():
    model = build_unet(
        spatial_dims=2,
        in_channels=1,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
    ).to(device)
    return model


def make_temporal_loader(stride: int):
    dataset = EchoNetTemporalVariableStrideDataset(
        eval_samples,
        videos_dir=VIDEOS_DIR,
        sequence_length=SEQUENCE_LENGTH,
        temporal_stride=stride,
        image_size=IMAGE_SIZE,
        augment=False,
        fps_by_video=fps_by_video,
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


def batch_item(batch, key, index=0, default=None):
    value = batch.get(key, default)
    if isinstance(value, torch.Tensor):
        selected = value[index]
        return selected.detach().cpu().tolist() if selected.ndim > 0 else selected.detach().cpu().item()
    if isinstance(value, (list, tuple)):
        return value[index]
    return value

## Run Grad-CAM evaluation

In [ ]:
per_sample_rows = []
overlay_sample_ids = set()


def should_save_overlay(sample_id: str) -> bool:
    if sample_id in overlay_sample_ids:
        return True
    if len(overlay_sample_ids) < MAX_OVERLAY_SAMPLES:
        overlay_sample_ids.add(sample_id)
        return True
    return False

def cleanup_gradcam_gpu(model=None):
    if model is not None:
        model.zero_grad(set_to_none=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def evaluate_convlstm_stride(stride: int):
    checkpoint_path = convlstm_checkpoint_for_stride(stride)
    if checkpoint_path is None or not checkpoint_path.exists():
        warnings.warn(f'Skipping ConvLSTM stride {stride}: checkpoint missing at {checkpoint_path}')
        return

    model, config = build_convlstm_for_stride(stride)
    load_state_dict_from_checkpoint(model, checkpoint_path)
    loader = make_temporal_loader(stride)
    model_id = f'convlstm_stride_{stride}'

    try:
        for batch in loader:
            sequence = batch['sequence'].to(device)
            center_mask = batch['mask'].to(device)
            try:
                sample_id = sanitize_name(batch_item(batch, 'id'))
                video_id = batch_item(batch, 'video_id')
                center_frame_idx = int(batch_item(batch, 'frame_idx'))
                frame_indices = batch_item(batch, 'frame_indices')
                fps = float(batch_item(batch, 'fps', default=float('nan')))
                window_span_seconds = float(batch_item(batch, 'window_span_seconds', default=float('nan')))
                frames_np = sequence[0, :, 0].detach().cpu().numpy().copy()
                center_mask_np = center_mask[0, 0].detach().cpu().numpy().copy()

                for layer_name in CONVLSTM_TARGET_LAYERS:
                    result = None
                    heatmaps = None
                    try:
                        result = convlstm_gradcam(model, sequence, center_mask, layer_name)
                        heatmaps = np.asarray(result.heatmaps, dtype=np.float32).copy()
                        n_layer_calls = result.n_layer_calls
                        result = None

                        prefix = f'{model_id}_{sanitize_name(layer_name)}_{sample_id}'
                        save_heatmaps(heatmaps, HEATMAP_DIR / model_id / sanitize_name(layer_name), prefix)
                        if should_save_overlay(sample_id):
                            save_overlay_grid(
                                frames_np,
                                heatmaps,
                                OVERLAY_DIR / model_id / sanitize_name(layer_name) / f'{prefix}_overlay.png',
                                title=f'{model_id} | {layer_name} | {sample_id}',
                            )
                        metadata = {
                            'sample_id': sample_id,
                            'video_id': video_id,
                            'center_frame_idx': center_frame_idx,
                            'frame_indices': ' '.join(str(x) for x in frame_indices),
                            'fps': fps,
                            'window_span_seconds': window_span_seconds,
                            'model_family': 'convlstm_unet',
                            'model_id': model_id,
                            'temporal_stride': stride,
                            'target_layer': layer_name,
                            'checkpoint_path': str(checkpoint_path),
                            'n_layer_calls_captured': n_layer_calls,
                        }
                        per_sample_rows.append(
                            compute_temporal_saliency_metrics(
                                heatmaps,
                                center_mask_np,
                                metadata=metadata,
                                saliency_percentile=SALIENCY_PERCENTILE,
                            )
                        )
                    finally:
                        del result, heatmaps
                        cleanup_gradcam_gpu(model)
            finally:
                del sequence, center_mask
                cleanup_gradcam_gpu(model)
    finally:
        del loader, model
        cleanup_gradcam_gpu()


def evaluate_unet_on_stride_sequences(stride: int, model, layer_name: str, checkpoint_path: Path):
    loader = make_temporal_loader(stride)
    model_id = f'unet2d_on_stride_{stride}_frames'
    try:
        for batch in loader:
            sequence = batch['sequence'].to(device)
            center_mask = batch['mask'].to(device)
            try:
                sample_id = sanitize_name(batch_item(batch, 'id'))
                video_id = batch_item(batch, 'video_id')
                center_frame_idx = int(batch_item(batch, 'frame_idx'))
                frame_indices = batch_item(batch, 'frame_indices')
                fps = float(batch_item(batch, 'fps', default=float('nan')))
                window_span_seconds = float(batch_item(batch, 'window_span_seconds', default=float('nan')))
                frames_np = sequence[0, :, 0].detach().cpu().numpy().copy()
                center_mask_np = center_mask[0, 0].detach().cpu().numpy().copy()

                result = None
                heatmaps = None
                try:
                    result = unet_framewise_gradcam(model, sequence, center_mask, layer_name)
                    heatmaps = np.asarray(result.heatmaps, dtype=np.float32).copy()
                    n_layer_calls = result.n_layer_calls
                    result = None

                    prefix = f'{model_id}_{sanitize_name(layer_name)}_{sample_id}'
                    save_heatmaps(heatmaps, HEATMAP_DIR / model_id / sanitize_name(layer_name), prefix)
                    if should_save_overlay(sample_id):
                        save_overlay_grid(
                            frames_np,
                            heatmaps,
                            OVERLAY_DIR / model_id / sanitize_name(layer_name) / f'{prefix}_overlay.png',
                            title=f'{model_id} | {layer_name} | {sample_id}',
                        )
                    metadata = {
                        'sample_id': sample_id,
                        'video_id': video_id,
                        'center_frame_idx': center_frame_idx,
                        'frame_indices': ' '.join(str(x) for x in frame_indices),
                        'fps': fps,
                        'window_span_seconds': window_span_seconds,
                        'model_family': 'unet_2d',
                        'model_id': model_id,
                        'temporal_stride': stride,
                        'target_layer': layer_name,
                        'checkpoint_path': str(checkpoint_path),
                        'n_layer_calls_captured': n_layer_calls,
                    }
                    per_sample_rows.append(
                        compute_temporal_saliency_metrics(
                            heatmaps,
                            center_mask_np,
                            metadata=metadata,
                            saliency_percentile=SALIENCY_PERCENTILE,
                        )
                    )
                finally:
                    del result, heatmaps
                    cleanup_gradcam_gpu(model)
            finally:
                del sequence, center_mask
                cleanup_gradcam_gpu(model)
    finally:
        del loader
        cleanup_gradcam_gpu(model)


In [ ]:
for stride in CONVLSTM_STRIDES:
    evaluate_convlstm_stride(stride)

if UNET_CHECKPOINT_PATH is None or not UNET_CHECKPOINT_PATH.exists():
    warnings.warn(f'Skipping 2D U-Net baseline: checkpoint missing at {UNET_CHECKPOINT_PATH}')
else:
    unet_model = build_unet_baseline()
    load_state_dict_from_checkpoint(unet_model, UNET_CHECKPOINT_PATH)
    unet_target_layer = find_last_conv2d_name(unet_model)
    print(f'2D U-Net Grad-CAM target layer: {unet_target_layer}')
    for stride in CONVLSTM_STRIDES:
        evaluate_unet_on_stride_sequences(stride, unet_model, unet_target_layer, UNET_CHECKPOINT_PATH)

per_sample_df = pd.DataFrame(per_sample_rows)
per_sample_df.to_csv(METRICS_DIR / 'per_sample_metrics.csv', index=False)
per_sample_df.to_csv(TABLES_DIR / 'per_sample_metrics.csv', index=False)
print(f'Per-sample rows: {len(per_sample_df):,}')
per_sample_df.head()

## Aggregate comparisons and plots

In [ ]:
aggregated_df = aggregate_metrics(per_sample_df)
aggregated_df.to_csv(METRICS_DIR / 'aggregated_metrics.csv', index=False)
aggregated_df.to_csv(TABLES_DIR / 'aggregated_metrics.csv', index=False)

if not per_sample_df.empty:
    stride_comparison = aggregated_df[aggregated_df['model_family'] == 'convlstm_unet'].copy()
    layer_comparison = aggregated_df.copy()
    model_comparison = aggregated_df.groupby(['model_family', 'target_layer'], dropna=False).mean(numeric_only=True).reset_index()
else:
    stride_comparison = pd.DataFrame()
    layer_comparison = pd.DataFrame()
    model_comparison = pd.DataFrame()

stride_comparison.to_csv(TABLES_DIR / 'comparison_across_strides.csv', index=False)
layer_comparison.to_csv(TABLES_DIR / 'comparison_across_target_layers.csv', index=False)
model_comparison.to_csv(TABLES_DIR / 'comparison_convlstm_vs_unet.csv', index=False)
save_metric_plots(aggregated_df, FIGURES_DIR)

summary = {
    'run_mode': RUN_MODE,
    'official_test_samples_available': len(test_samples),
    'evaluation_samples': len(eval_samples),
    'convlstm_target_layers': CONVLSTM_TARGET_LAYERS,
    'convlstm_strides': CONVLSTM_STRIDES,
    'saliency_percentile': SALIENCY_PERCENTILE,
    'max_overlay_samples': MAX_OVERLAY_SAMPLES,
    'overlay_samples_saved': len(overlay_sample_ids),
    'notes': 'Neighbor-frame ground-truth LV masks are not available in the processed center-frame dataset. Temporal metrics are computed only from CAM heatmaps; LV overlap is computed only between the center-frame CAM and the center-frame LV mask.',
}
with (OUTPUT_DIR / 'gradcam_temporal_evaluation_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

aggregated_df

## Output checks

In [ ]:
required_outputs = [
    METRICS_DIR / 'per_sample_metrics.csv',
    METRICS_DIR / 'aggregated_metrics.csv',
    TABLES_DIR / 'comparison_across_strides.csv',
    TABLES_DIR / 'comparison_across_target_layers.csv',
    TABLES_DIR / 'comparison_convlstm_vs_unet.csv',
    TABLES_DIR / 'checkpoint_discovery.csv',
    OUTPUT_DIR / 'gradcam_temporal_evaluation_summary.json',
]
missing = [path for path in required_outputs if not path.exists()]
assert not missing, f'Missing expected outputs: {missing}'
print(f'Grad-CAM temporal evaluation outputs saved to: {OUTPUT_DIR.resolve()}')